# Cálculo da Taxa por 100 Mil Habitantes 

$$\text{Taxa}_{regional} = \frac{\sum \text{Quantidade dos eventos dos municípios da regional}}{\sum \text{População dos municípios da regional}} \times 100.000$$



### 1. Imports


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# utilitarios_comuns.py fica na pasta acima deste notebook
sys.path.append(str(Path.cwd().parent))

from utilitarios_comuns import (
    registrar_log,
    normalizar_codigo_ibge,
    normalizar_regional,
    extrair_ano_da_coluna,
    selecionar_colunas_de_valor,
    filtrar_populacao_valida_por_municipio,
    COLUNA_MUNICIPIO,
    COLUNA_CODIGO_IBGE,
    COLUNA_REGIONAL,
    COLUNA_TIPO,
    POPULACAO_MINIMA_VALIDA,
    SUFIXO_COLUNA_TAXA,
    PASTA_POPULACAO,
    CAMINHO_POPULACAO,
    SUFIXO_COLUNA_TOTAL,
    FAIXAS_ETARIAS_PADRAO,
    MULTIPLICADOR_TAXA,
    ESPECIFICACOES_VARIAVEIS,
    ESPECIFICACOES_VARIAVEIS_CATEGORICAS,
    SUFIXO_VARIAVEL_ESTADUAL,
    NOME_ARQUIVO_AGREGADO_CATEGORICO_POR_ANO,
)

### Configurações

In [2]:
CASAS_DECIMAIS_TAXA = 2

PASTA_SAIDA = PASTA_POPULACAO / "Taxa"

LISTA_VARIAVEIS = []
for nome, especificacao in ESPECIFICACOES_VARIAVEIS.items():
    if nome == "Hospitais":
        continue  # Hospitais tem processamento à parte (agrupa por TIPO também)
    configuracao_da_variavel = {"NOME": nome, **especificacao}
    configuracao_da_variavel["NOME_ARQUIVO_SAIDA_REGIONAL"] = especificacao["NOME_ARQUIVO_SAIDA_TAXA"]
    LISTA_VARIAVEIS.append(configuracao_da_variavel)

LISTA_VARIAVEIS_CATEGORICAS = []

for nome, especificacao in ESPECIFICACOES_VARIAVEIS_CATEGORICAS.items():
    configuracao_da_variavel = {"NOME": nome, **especificacao}
    configuracao_da_variavel["NOME_ARQUIVO_SAIDA_REGIONAL"] = especificacao["NOME_ARQUIVO_SAIDA_TAXA"]
    LISTA_VARIAVEIS_CATEGORICAS.append(configuracao_da_variavel)

CAMINHO_HOSPITAIS = ESPECIFICACOES_VARIAVEIS["Hospitais"]["CAMINHO_ENTRADA"]
ANO_REFERENCIA_HOSPITAIS = ESPECIFICACOES_VARIAVEIS["Hospitais"]["ANO_INICIAL"]
NOME_ARQUIVO_SAIDA_HOSPITAIS_REGIONAL = ESPECIFICACOES_VARIAVEIS["Hospitais"]["NOME_ARQUIVO_SAIDA_TAXA"]

### Carregamento da População (uma única vez)

In [3]:
tabela_populacao = pd.read_csv(CAMINHO_POPULACAO)
tabela_populacao[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_populacao[COLUNA_CODIGO_IBGE])
tabela_populacao[COLUNA_REGIONAL] = normalizar_regional(tabela_populacao[COLUNA_REGIONAL])

registrar_log(f"População carregada: {tabela_populacao.shape[0]} municípios.")
tabela_populacao.head()

[LOG] População carregada: 853 municípios.


,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1970_0-9,1970_10-19,1970_20-29,1970_30-39,1970_40-49,1970_50-59,1970_60-69,...,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL,2024_TOTAL,2025_TOTAL
0,abadia dos dourados,3100104,uberlandia,1542.0,1255.0,791.0,537.0,274.0,207.0,143.0,...,3277.0,3233.0,3195.0,3164.0,3142.0,3123.0,3111.0,3105.0,3106.0,3115.0
1,abaete,3100203,sete lagoas,3054.0,2786.0,1423.0,1115.0,874.0,470.0,311.0,...,11149.0,11177.0,11215.0,11263.0,11322.0,11392.0,11474.0,11569.0,11677.0,11799.0
2,abre campo,3100302,manhumirim,3015.0,2534.0,1462.0,1049.0,678.0,543.0,311.0,...,6956.0,6933.0,6914.0,6901.0,6895.0,6895.0,6903.0,6917.0,6936.0,6963.0
3,acaiaca,3100401,ponte nova,514.0,387.0,200.0,208.0,135.0,128.0,70.0,...,1918.0,1930.0,1942.0,1955.0,1974.0,1992.0,2012.0,2035.0,2057.0,2084.0
4,aguanil,3100807,divinopolis,359.0,302.0,217.0,187.0,126.0,75.0,23.0,...,2042.0,2065.0,2087.0,2112.0,2139.0,2167.0,2196.0,2228.0,2261.0,2297.0


## Calcular a taxa — agregando por Regional de Saúde antes de dividir

Dois cruzamentos, em pontos diferentes do processamento:

- **Por Código IBGE**, entre a base bruta de cada variável e a
  população: valida o par (município, ano) e é o que permite
  detectar e descartar, por ano, um município sem população válida
  sem descartar os demais anos do mesmo município (ver
  `pares_descartados` abaixo).
- **Por nome da Regional de Saúde** (já normalizado por
  `normalizar_regional`), depois de já ter somado quantidade e
  população por município: agrega ambas as somas ao nível de
  Regional antes de dividir, para que o resultado seja a razão das
  somas regionais, não uma soma de taxas municipais já calculadas.


In [ ]:
def calcular_taxa(quantidade, populacao):
    """Taxa por 100 mil habitantes: quantidade / população válida (>= POPULACAO_MINIMA_VALIDA) × MULTIPLICADOR_TAXA."""
    populacao_valida = populacao.where(populacao >= POPULACAO_MINIMA_VALIDA)
    return (quantidade / populacao_valida) * MULTIPLICADOR_TAXA


def coluna_populacao_correspondente(coluna: str, ano: int) -> str:
    """Devolve a coluna de população do denominador: a da mesma faixa etária e ano, ou a total da regional."""
    prefixo = f"{ano}_"
    categoria = None
    if str(coluna).startswith(prefixo):
        categoria = str(coluna)[len(prefixo):]
    if categoria in FAIXAS_ETARIAS_PADRAO:
        return f"{ano}_{categoria}"
    return f"{ano}{SUFIXO_COLUNA_TOTAL}"


def calcular_taxa_por_regional(tabela_bruta, tabela_populacao, ano_inicial, ano_final,
                                colunas_de_agrupamento_extra=(), nome_da_variavel=""):
    """Agrega quantidade e população por Regional de Saúde (e colunas extra, se houver) e devolve a taxa por 100k hab. de cada coluna do período."""
    colunas_de_agrupamento = [COLUNA_REGIONAL, *colunas_de_agrupamento_extra]

    municipios_sem_regional = tabela_bruta[COLUNA_REGIONAL].isna().sum()
    if municipios_sem_regional > 0:
        registrar_log(
            f"ATENÇÃO: {municipios_sem_regional} registro(s) sem Regional de Saúde "
            f"-- excluído(s) da agregação por regional."
        )
    tabela_bruta = tabela_bruta.dropna(subset=[COLUNA_REGIONAL]).copy()

    colunas_de_valor = selecionar_colunas_de_valor(
        tabela_bruta, ano_inicial, ano_final,
        colunas_a_ignorar=(COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE, *colunas_de_agrupamento),
    )

    tabela_bruta[colunas_de_valor] = tabela_bruta[colunas_de_valor].astype(float)

    # Coluna de população de cada coluna de valor: da mesma faixa e ano, ou a total da regional
    coluna_para_populacao = {}
    for coluna in colunas_de_valor:
        coluna_para_populacao[coluna] = coluna_populacao_correspondente(coluna, extrair_ano_da_coluna(coluna))

    # População somada por regional e ano, inclusive a de cada faixa etária
    colunas_de_populacao_encontradas = set()
    for coluna_populacao in coluna_para_populacao.values():
        if coluna_populacao in tabela_populacao.columns:
            colunas_de_populacao_encontradas.add(coluna_populacao)
    colunas_de_populacao_no_periodo = sorted(colunas_de_populacao_encontradas)

    tabela_populacao_filtrada = filtrar_populacao_valida_por_municipio(
        tabela_populacao, colunas_de_populacao_no_periodo
    )

    if tabela_populacao_filtrada[COLUNA_CODIGO_IBGE].duplicated().any():
        raise ValueError(
            "tabela_populacao contém CÓDIGO IBGE duplicado -- o alinhamento por "
            "código IBGE (.map) exige códigos únicos."
        )
    tabela_populacao_filtrada_indexada = tabela_populacao_filtrada.set_index(COLUNA_CODIGO_IBGE)

    # Agrupa por (ano, coluna de população): faixas etárias têm população própria, verificada em grupos separados
    colunas_por_pop = {}
    for coluna in colunas_de_valor:
        ano = extrair_ano_da_coluna(coluna)
        chave = (ano, coluna_para_populacao[coluna])
        colunas_por_pop.setdefault(chave, []).append(coluna)

    pares_descartados = 0
    eventos_descartados = 0.0
    anos_afetados = []

    for (ano, coluna_populacao_do_ano), colunas_do_grupo in colunas_por_pop.items():
        if coluna_populacao_do_ano not in tabela_populacao_filtrada_indexada.columns:
            continue
        populacao_do_municipio_no_ano = tabela_bruta[COLUNA_CODIGO_IBGE].map(
            tabela_populacao_filtrada_indexada[coluna_populacao_do_ano]
        )
        mascara_populacao_invalida = populacao_do_municipio_no_ano.isna()
        if not mascara_populacao_invalida.any():
            continue
        mascara_com_quantidade = mascara_populacao_invalida & tabela_bruta[colunas_do_grupo].notna().any(axis=1)
        if mascara_com_quantidade.any():
            if coluna_populacao_do_ano in colunas_do_grupo:
                colunas_para_contagem = [coluna_populacao_do_ano]
            else:
                colunas_para_contagem = colunas_do_grupo
            pares_descartados += int(mascara_com_quantidade.sum())
            eventos_descartados += float(
                tabela_bruta.loc[mascara_com_quantidade, colunas_para_contagem].sum().sum()
            )
            anos_afetados.append(ano)
        tabela_bruta.loc[mascara_populacao_invalida, colunas_do_grupo] = np.nan

    if pares_descartados > 0 and nome_da_variavel:
        rotulo_par = "(" + ", ".join(["município", *colunas_de_agrupamento_extra, "ano"]) + ")"
        registrar_log(
            f"{nome_da_variavel}: {pares_descartados} par(es) {rotulo_par} com quantidade "
            f"descartada por população inválida antes da soma por regional -- "
            f"{eventos_descartados:g} evento(s) no total; ano(s) afetado(s): {sorted(set(anos_afetados))}."
        )

    quantidade_por_regional = (
        tabela_bruta.groupby(colunas_de_agrupamento, as_index=False)[colunas_de_valor]
        .sum(min_count=1)  # se todos os municípios são NaN, o resultado continua NaN
    )

    populacao_por_regional = (
        tabela_populacao_filtrada.groupby(COLUNA_REGIONAL)[colunas_de_populacao_no_periodo]
        .sum(min_count=1)
    )

    colunas_de_taxa_calculadas = {}
    for coluna in colunas_de_valor:
        coluna_populacao_do_ano = coluna_para_populacao[coluna]
        ano = extrair_ano_da_coluna(coluna)

        if coluna_populacao_do_ano not in populacao_por_regional.columns:
            registrar_log(
                f"ATENÇÃO: sem população para o ano {ano} -- coluna '{coluna}' ficará vazia (NaN)."
            )
            colunas_de_taxa_calculadas[coluna] = np.nan
            quantidade_por_regional[coluna] = np.nan
            continue

        populacao_da_regional_no_ano = quantidade_por_regional[COLUNA_REGIONAL].map(
            populacao_por_regional[coluna_populacao_do_ano]
        )
        taxa_da_coluna = calcular_taxa(quantidade_por_regional[coluna], populacao_da_regional_no_ano)
        colunas_de_taxa_calculadas[coluna] = taxa_da_coluna

    tabela_de_taxas = pd.DataFrame(colunas_de_taxa_calculadas).round(CASAS_DECIMAIS_TAXA)

    blocos = [quantidade_por_regional[colunas_de_agrupamento]]

    for coluna in colunas_de_valor:
        blocos.append(quantidade_por_regional[[coluna]])
        blocos.append(
            tabela_de_taxas[[coluna]].rename(columns={coluna: f"{coluna}{SUFIXO_COLUNA_TAXA}"})
        )
    return pd.concat(blocos, axis=1)


def calcular_com_total_por_tipo(funcao_calculo, tabela_bruta, tabela_populacao, ano_inicial, ano_final,
                                 nome_da_variavel, colunas_de_ordenacao=None):
    """Roda o cálculo agrupando por TIPO e sem agrupar (linha com TIPO='TODOS') e concatena os resultados."""
    tabela_por_tipo = funcao_calculo(
        tabela_bruta, tabela_populacao, ano_inicial, ano_final,
        colunas_de_agrupamento_extra=(COLUNA_TIPO,),
        nome_da_variavel=nome_da_variavel,
    )

    tabela_total = funcao_calculo(
        tabela_bruta, tabela_populacao, ano_inicial, ano_final,
        nome_da_variavel=f"{nome_da_variavel} (Total)",
    )
    tabela_total.insert(1, COLUNA_TIPO, "TODOS")

    tabela_combinada = pd.concat([tabela_por_tipo, tabela_total], ignore_index=True)
    if colunas_de_ordenacao:
        tabela_combinada = tabela_combinada.sort_values(colunas_de_ordenacao).reset_index(drop=True)
    return tabela_combinada


### Variáveis — Casos, Mortalidade, Mamografia, Escolaridade, Raça/Cor, Faixa Etária

Mesmo processamento pras variáveis simples e categóricas: só muda a
lista de configuração percorrida (a diferença entre elas está nas
colunas de valor de cada CSV de entrada, não no código).

In [5]:
tabelas_brutas_normalizadas = {}

for configuracao_da_variavel in LISTA_VARIAVEIS + LISTA_VARIAVEIS_CATEGORICAS:
    nome_da_variavel = configuracao_da_variavel["NOME"]
    registrar_log(f"===== INÍCIO: {nome_da_variavel.upper()} =====")

    tabela_bruta = pd.read_csv(configuracao_da_variavel["CAMINHO_ENTRADA"])
    tabela_bruta[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_bruta[COLUNA_CODIGO_IBGE])
    tabela_bruta[COLUNA_REGIONAL] = normalizar_regional(tabela_bruta[COLUNA_REGIONAL])
    # Guardada para o agregado estadual, sem reler o CSV
    tabelas_brutas_normalizadas[nome_da_variavel] = tabela_bruta

    tabela_com_taxa_regional = calcular_taxa_por_regional(
        tabela_bruta, tabela_populacao,
        configuracao_da_variavel["ANO_INICIAL"], configuracao_da_variavel["ANO_FINAL"],
        nome_da_variavel=nome_da_variavel,
    )

    caminho_de_saida_regional = PASTA_SAIDA / configuracao_da_variavel["NOME_ARQUIVO_SAIDA_REGIONAL"]
    caminho_de_saida_regional.parent.mkdir(parents=True, exist_ok=True)
    tabela_com_taxa_regional.to_csv(caminho_de_saida_regional, index=False)

    registrar_log(
        f"===== FIM: {nome_da_variavel.upper()} — salvo em {caminho_de_saida_regional.name} =====\n"
    )

[LOG] ===== INÍCIO: CASOS =====
[LOG] ATENÇÃO: 130 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.
[LOG] Casos: 698 par(es) (município, ano) com quantidade descartada por população inválida antes da soma por regional -- 0 evento(s) no total; ano(s) afetado(s): [1985, 1986, 1987, 1988, 1989, 1990].
[LOG] ===== FIM: CASOS — salvo em tabela_casos_por_ano_de_diagnóstico_taxa.csv =====

[LOG] ===== INÍCIO: MORTALIDADE =====


[LOG] ATENÇÃO: 134 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.


[LOG] Mortalidade: 97 par(es) (município, ano) com quantidade descartada por população inválida antes da soma por regional -- 98 evento(s) no total; ano(s) afetado(s): [1994, 1995, 1996, 1997, 1998, 1999, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2023, 2024].


/tmp/ipykernel_98579/2178142776.py:102: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # se todos os municípios são NaN, o resultado continua NaN


[LOG] ===== FIM: MORTALIDADE — salvo em tabela_mortes_por_faixa_etária_taxa.csv =====

[LOG] ===== INÍCIO: MAMOGRAFIA =====
[LOG] ATENÇÃO: 131 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.
[LOG] Mamografia: 1149 par(es) (município, ano) com quantidade descartada por população inválida antes da soma por regional -- 2805 evento(s) no total; ano(s) afetado(s): [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2023, 2024, 2025].
[LOG] ===== FIM: MAMOGRAFIA — salvo em tabela_mamografias_por_faixa_etária_taxa.csv =====

[LOG] ===== INÍCIO: CASOS POR ESCOLARIDADE =====


/tmp/ipykernel_98579/2178142776.py:102: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # se todos os municípios são NaN, o resultado continua NaN
/tmp/ipykernel_98579/2178142776.py:102: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # se todos os municípios são NaN, o resultado continua NaN


... (2 outputs left out)

[LOG] ===== FIM: CASOS POR RAÇA E COR — salvo em tabela_casos_por_raça_e_cor_taxa.csv =====

[LOG] ===== INÍCIO: CASOS POR FAIXA ETÁRIA =====
[LOG] ATENÇÃO: 133 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.


[LOG] Casos por Faixa Etária: 1246 par(es) (município, ano) com quantidade descartada por população inválida antes da soma por regional -- 288 evento(s) no total; ano(s) afetado(s): [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2023].


/tmp/ipykernel_98579/2178142776.py:102: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # se todos os municípios são NaN, o resultado continua NaN


[LOG] ===== FIM: CASOS POR FAIXA ETÁRIA — salvo em tabela_casos_por_faixa_etária_taxa.csv =====



### Variável Hospital

Agrupa por (`REGIONAL DE SAÚDE`, `TIPO`) em vez de só `REGIONAL DE
SAÚDE` — cada tipo de habilitação continua em uma linha própria, sem
somar Oncologia com Reconstrução Mamária.

O mesmo CSV de saída também recebe uma linha por regional com
`TIPO = "TODOS"` (soma de todos os tipos de habilitação). Os
gráficos filtram por `TIPO` antes de somar; qualquer outro consumo
deste CSV precisa filtrar `TIPO` também, ou vai contar cada hospital
em dobro (uma vez por tipo específico, outra em "TODOS").

In [6]:
tabela_hospitais_bruta = pd.read_csv(CAMINHO_HOSPITAIS)
tabela_hospitais_bruta[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_hospitais_bruta[COLUNA_CODIGO_IBGE])
tabela_hospitais_bruta[COLUNA_REGIONAL] = normalizar_regional(tabela_hospitais_bruta[COLUNA_REGIONAL])

tabela_hospitais_com_taxa_regional = calcular_com_total_por_tipo(
    calcular_taxa_por_regional, tabela_hospitais_bruta, tabela_populacao,
    ANO_REFERENCIA_HOSPITAIS, ANO_REFERENCIA_HOSPITAIS,
    nome_da_variavel="Hospitais",
    colunas_de_ordenacao=[COLUNA_REGIONAL, COLUNA_TIPO],
)

caminho_de_saida_hospitais_regional = PASTA_SAIDA / NOME_ARQUIVO_SAIDA_HOSPITAIS_REGIONAL
caminho_de_saida_hospitais_regional.parent.mkdir(parents=True, exist_ok=True)
tabela_hospitais_com_taxa_regional.to_csv(caminho_de_saida_hospitais_regional, index=False)

registrar_log(f"Hospitais salvo em {caminho_de_saida_hospitais_regional.name}")
tabela_hospitais_com_taxa_regional.head()

[LOG] Hospitais salvo em tabela_hospitais_habilitados_taxa.csv


,REGIONAL DE SAÚDE,TIPO,2023_TOTAL,2023_TOTAL_TAXA
0,alfenas,ONCOLOGIA,1.0,0.42
1,alfenas,RECONSTRUÇÃO MAMÁRIA,1.0,0.42
2,alfenas,TODOS,2.0,0.84
3,barbacena,ONCOLOGIA,1.0,0.38
4,barbacena,TODOS,1.0,0.38


### Agregado Estadual por Ano — Todas as Variáveis

Taxa e quantidade **estaduais** (soma de todas as Regionais) de
todas as variáveis -- Casos, Mortalidade, Mamografia (cada uma com
sua quebra por Faixa Etária quando existir), Hospitais, além de
Escolaridade e Raça/Cor de Casos -- eram recalculadas em tempo real
em `Avaliação e Apresentação` (`agregar_categorica_por_ano`) para o
gráfico de Casos por categoria -- fora de lugar no KDD, já que é
cálculo, não apresentação. Movido para cá.

`calcular_taxa_estadual` reusa `calcular_taxa_por_regional` já
validada (sobrescrevendo REGIONAL DE SAÚDE com um valor único), sem
duplicar a lógica de população específica por faixa etária nem a
de descarte de município com população inválida.

Formato final -- **uma linha por ANO** (1979-2025, união dos
períodos de cada variável), colunas "`<categoria>_<VARIÁVEL>`" e
"`<categoria>_<VARIÁVEL>_TAXA`" (ex.: `0-9_MORTES`, `TOTAL_CASOS`).
O sufixo da variável (`SUFIXO_VARIAVEL_ESTADUAL`, em
`utilitarios_comuns.py`) evita colisão entre categorias do mesmo nome
em variáveis diferentes; Escolaridade e Raça/Cor (ambas de Casos)
usam sufixos próprios (`CASOS_ESCOLARIDADE`/`CASOS_RACA_COR`) porque
as duas têm uma categoria "SEM INFORMAÇÃO" -- colidiria sob "CASOS"
sozinho.

Duas exclusões evitam coluna duplicada: "DIAGNOSTICO" em Casos
(idêntico a TOTAL nessa fonte, conferido -- sempre o mesmo valor) e
"TOTAL" nas 3 quebras categóricas de Casos (já coberto pela fonte
dedicada "Casos", que é a soma real -- as 3 quebras teriam cada uma
seu próprio TOTAL, redundante).

Hospitais foge do padrão (só 2023, categorizado por TIPO em vez de
uma quebra demográfica) -- tratado à parte, reaproveitando
`tabela_hospitais_bruta` já carregada/normalizada na seção anterior.

In [7]:
def calcular_taxa_estadual(tabela_bruta, tabela_populacao, ano_inicial, ano_final,
                            colunas_de_agrupamento_extra=(), nome_da_variavel=""):
    """Mesmo cálculo de calcular_taxa_por_regional, com o estado inteiro como uma única Regional ('ESTADUAL')."""
    tabela_bruta_estadual = tabela_bruta.copy()
    tabela_bruta_estadual[COLUNA_REGIONAL] = "ESTADUAL"
    tabela_populacao_estadual = tabela_populacao.copy()
    tabela_populacao_estadual[COLUNA_REGIONAL] = "ESTADUAL"
    return calcular_taxa_por_regional(
        tabela_bruta_estadual, tabela_populacao_estadual, ano_inicial, ano_final,
        colunas_de_agrupamento_extra=colunas_de_agrupamento_extra,
        nome_da_variavel=nome_da_variavel,
    )


def _formato_longo_ano_categoria(tabela_taxa_estadual):
    """Converte a linha única da taxa estadual em formato longo: uma linha por (ANO, CATEGORIA)."""
    linha = tabela_taxa_estadual.iloc[0]
    registros = {}
    for coluna, valor in linha.items():
        if coluna == COLUNA_REGIONAL:
            continue
        eh_taxa = coluna.endswith(SUFIXO_COLUNA_TAXA)
        base = coluna
        if eh_taxa:
            base = coluna[: -len(SUFIXO_COLUNA_TAXA)]
        ano_str, categoria = base.split("_", 1)
        registro = registros.setdefault((int(ano_str), categoria), {"ANO": int(ano_str), "CATEGORIA": categoria})
        registro["TAXA" if eh_taxa else "QUANTIDADE"] = valor
    return pd.DataFrame(registros.values())


tabelas_longas = []

for configuracao_da_variavel in LISTA_VARIAVEIS:
    nome_da_variavel = configuracao_da_variavel["NOME"]
    if nome_da_variavel == "Hospitais":
        continue  # tratado à parte, mais abaixo
    sufixo = SUFIXO_VARIAVEL_ESTADUAL[nome_da_variavel]
    # 'DIAGNOSTICO' é idêntico a TOTAL em Casos; mantém só TOTAL
    categorias_a_excluir = ()
    if nome_da_variavel == "Casos":
        categorias_a_excluir = ("DIAGNOSTICO",)

    # Reaproveita a tabela já lida e normalizada no loop principal
    tabela_bruta = tabelas_brutas_normalizadas[nome_da_variavel]

    tabela_taxa_estadual = calcular_taxa_estadual(
        tabela_bruta, tabela_populacao,
        configuracao_da_variavel["ANO_INICIAL"], configuracao_da_variavel["ANO_FINAL"],
        nome_da_variavel=f"{nome_da_variavel} (Estadual)",
    )
    tabela_longa = _formato_longo_ano_categoria(tabela_taxa_estadual)
    tabela_longa = tabela_longa[~tabela_longa["CATEGORIA"].isin(categorias_a_excluir)].copy()
    tabela_longa["CATEGORIA_VARIAVEL"] = tabela_longa["CATEGORIA"] + f"_{sufixo}"
    tabelas_longas.append(tabela_longa[["ANO", "CATEGORIA_VARIAVEL", "QUANTIDADE", "TAXA"]])
    registrar_log(f"{nome_da_variavel}: agregado estadual calculado.")

for configuracao_da_variavel in LISTA_VARIAVEIS_CATEGORICAS:
    nome_da_variavel = configuracao_da_variavel["NOME"]
    sufixo = SUFIXO_VARIAVEL_ESTADUAL[nome_da_variavel]

    # Reaproveita a tabela já lida e normalizada no loop principal
    tabela_bruta = tabelas_brutas_normalizadas[nome_da_variavel]

    tabela_taxa_estadual = calcular_taxa_estadual(
        tabela_bruta, tabela_populacao,
        configuracao_da_variavel["ANO_INICIAL"], configuracao_da_variavel["ANO_FINAL"],
        nome_da_variavel=f"{nome_da_variavel} (Estadual)",
    )
    tabela_longa = _formato_longo_ano_categoria(tabela_taxa_estadual)
    # Descarta 'TOTAL', que já vem da variável Casos
    tabela_longa = tabela_longa[tabela_longa["CATEGORIA"] != "TOTAL"].copy()
    tabela_longa["CATEGORIA_VARIAVEL"] = tabela_longa["CATEGORIA"] + f"_{sufixo}"
    tabelas_longas.append(tabela_longa[["ANO", "CATEGORIA_VARIAVEL", "QUANTIDADE", "TAXA"]])
    registrar_log(f"{nome_da_variavel}: agregado estadual calculado.")

# Hospitais: um único ano, categorizado por TIPO
tabela_hospitais_estadual = calcular_com_total_por_tipo(
    calcular_taxa_estadual, tabela_hospitais_bruta, tabela_populacao,
    ANO_REFERENCIA_HOSPITAIS, ANO_REFERENCIA_HOSPITAIS,
    nome_da_variavel="Hospitais (Estadual)",
)

sufixo_hospitais = SUFIXO_VARIAVEL_ESTADUAL["Hospitais"]
coluna_quantidade_hospitais = f"{ANO_REFERENCIA_HOSPITAIS}{SUFIXO_COLUNA_TOTAL}"
coluna_taxa_hospitais = f"{coluna_quantidade_hospitais}{SUFIXO_COLUNA_TAXA}"
registros_hospitais = []
for _, linha in tabela_hospitais_estadual.iterrows():
    registros_hospitais.append({
        "ANO": ANO_REFERENCIA_HOSPITAIS,
        "CATEGORIA_VARIAVEL": f"{linha[COLUNA_TIPO]}_{sufixo_hospitais}",
        "QUANTIDADE": linha[coluna_quantidade_hospitais],
        "TAXA": linha[coluna_taxa_hospitais],
    })
tabelas_longas.append(pd.DataFrame(registros_hospitais))
registrar_log("Hospitais: agregado estadual calculado.")

# Uma linha por ANO, com colunas '<categoria>_<variável>' e '..._TAXA'
tabela_longa_completa = pd.concat(tabelas_longas, ignore_index=True)
tabela_larga_por_ano = tabela_longa_completa.pivot(
    index="ANO", columns="CATEGORIA_VARIAVEL", values=["QUANTIDADE", "TAXA"]
)
nomes_das_colunas = []
for tipo, categoria_variavel in tabela_larga_por_ano.columns:
    if tipo == "QUANTIDADE":
        nomes_das_colunas.append(categoria_variavel)
    else:
        nomes_das_colunas.append(f"{categoria_variavel}{SUFIXO_COLUNA_TAXA}")
tabela_larga_por_ano.columns = nomes_das_colunas
tabela_regional_agregada_por_ano = tabela_larga_por_ano.reset_index().sort_values("ANO").reset_index(drop=True)

caminho_de_saida_agregado = PASTA_SAIDA / NOME_ARQUIVO_AGREGADO_CATEGORICO_POR_ANO
caminho_de_saida_agregado.parent.mkdir(parents=True, exist_ok=True)
tabela_regional_agregada_por_ano.to_csv(caminho_de_saida_agregado, index=False)

registrar_log(
    f"Agregado estadual por ano salvo em {caminho_de_saida_agregado.name} "
    f"({tabela_regional_agregada_por_ano['ANO'].min()}-{tabela_regional_agregada_por_ano['ANO'].max()}, "
    f"{tabela_regional_agregada_por_ano.shape[1] - 1} colunas)."
)
tabela_regional_agregada_por_ano.head()

[LOG] ATENÇÃO: 130 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.
[LOG] Casos (Estadual): 698 par(es) (município, ano) com quantidade descartada por população inválida antes da soma por regional -- 0 evento(s) no total; ano(s) afetado(s): [1985, 1986, 1987, 1988, 1989, 1990].
[LOG] Casos: agregado estadual calculado.
[LOG] ATENÇÃO: 134 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.


[LOG] Mortalidade (Estadual): 97 par(es) (município, ano) com quantidade descartada por população inválida antes da soma por regional -- 98 evento(s) no total; ano(s) afetado(s): [1994, 1995, 1996, 1997, 1998, 1999, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2023, 2024].


/tmp/ipykernel_98579/2178142776.py:102: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # se todos os municípios são NaN, o resultado continua NaN


[LOG] Mortalidade: agregado estadual calculado.
[LOG] ATENÇÃO: 131 município(s) sem população válida (< 1) em ao menos um ano do período -- excluído(s) da soma por regional nesses anos.
[LOG] Mamografia (Estadual): 1149 par(es) (município, ano) com quantidade descartada por população inválida antes da soma por regional -- 2805 evento(s) no total; ano(s) afetado(s): [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2023, 2024, 2025].
[LOG] Mamografia: agregado estadual calculado.


/tmp/ipykernel_98579/2178142776.py:102: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # se todos os municípios são NaN, o resultado continua NaN
/tmp/ipykernel_98579/2178142776.py:102: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # se todos os municípios são NaN, o resultado continua NaN


[LOG] Casos por Escolaridade: agregado estadual calculado.
[LOG] Casos por Raça e Cor: agregado estadual calculado.


... (2 outputs left out)

[LOG] Casos por Faixa Etária (Estadual): 1246 par(es) (município, ano) com quantidade descartada por população inválida antes da soma por regional -- 288 evento(s) no total; ano(s) afetado(s): [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2023].


/tmp/ipykernel_98579/2178142776.py:102: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .sum(min_count=1)  # se todos os municípios são NaN, o resultado continua NaN


[LOG] Casos por Faixa Etária: agregado estadual calculado.
[LOG] Hospitais: agregado estadual calculado.
[LOG] Agregado estadual por ano salvo em tabela_regional_agregada_por_ano.csv (1979-2025, 88 colunas).


,ANO,0-9_CASOS,0-9_MAMOGRAFIAS,0-9_MORTES,10-19_CASOS,10-19_MAMOGRAFIAS,10-19_MORTES,20-29_CASOS,20-29_MAMOGRAFIAS,20-29_MORTES,...,PARDA_CASOS_RACA_COR_TAXA,PRETA_CASOS_RACA_COR_TAXA,RECONSTRUÇÃO MAMÁRIA_HOSPITAIS_TAXA,SEM INFORMAÇÃO_CASOS_ESCOLARIDADE_TAXA,SEM INFORMAÇÃO_CASOS_RACA_COR_TAXA,TODOS_HOSPITAIS_TAXA,TOTAL_CASOS_TAXA,TOTAL_MAMOGRAFIAS_TAXA,TOTAL_MORTES_TAXA,TRATAMENTO SINCRÔNICO_HOSPITAIS_TAXA
0,1979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.45,NaN
1,1980,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.70,NaN
2,1981,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.05,NaN
3,1982,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.70,NaN
4,1983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.29,NaN


### Tabela para o Overleaf — Numerador e Denominador por Variável

Resume, para cada variável processada acima, qual coluna entrou como
numerador e qual população entrou como denominador em
`calcular_taxa_por_regional` -- útil como referência ao escrever a
metodologia, sem precisar repetir cada uma das operações feitas nas
células anteriores.

In [8]:
def periodo(nome_da_variavel, especificacoes):
    especificacao_da_variavel = especificacoes[nome_da_variavel]
    return f"{especificacao_da_variavel['ANO_INICIAL']}–{especificacao_da_variavel['ANO_FINAL']}"


linhas_tabela_taxa = [
    {
        "Variável": "Casos (Ano de Diagnóstico)",
        "Numerador": f"Casos, por ano ({periodo('Casos', ESPECIFICACOES_VARIAVEIS)})",
        "Denominador": "População total da regional",
        "Agregação Extra": "--",
    },
    {
        "Variável": "Casos por Faixa Etária",
        "Numerador": f"Casos, por ano e faixa etária ({periodo('Casos por Faixa Etária', ESPECIFICACOES_VARIAVEIS_CATEGORICAS)})",
        "Denominador": "População da mesma faixa etária e regional",
        "Agregação Extra": "--",
    },
    {
        "Variável": "Casos por Escolaridade",
        "Numerador": f"Casos, por ano e escolaridade ({periodo('Casos por Escolaridade', ESPECIFICACOES_VARIAVEIS_CATEGORICAS)})",
        "Denominador": "População total da regional",
        "Agregação Extra": "--",
    },
    {
        "Variável": "Casos por Raça e Cor",
        "Numerador": f"Casos, por ano e raça/cor ({periodo('Casos por Raça e Cor', ESPECIFICACOES_VARIAVEIS_CATEGORICAS)})",
        "Denominador": "População total da regional",
        "Agregação Extra": "--",
    },
    {
        "Variável": "Mamografia por Faixa Etária",
        "Numerador": f"Mamografias, por ano e faixa etária ({periodo('Mamografia', ESPECIFICACOES_VARIAVEIS)})",
        "Denominador": "População da mesma faixa etária e regional",
        "Agregação Extra": "--",
    },
    {
        "Variável": "Mortalidade por Faixa Etária",
        "Numerador": f"Óbitos, por ano e faixa etária ({periodo('Mortalidade', ESPECIFICACOES_VARIAVEIS)})",
        "Denominador": "População da mesma faixa etária e regional",
        "Agregação Extra": "--",
    },
    {
        "Variável": "Hospitais Habilitados",
        "Numerador": f"Hospitais habilitados ({ANO_REFERENCIA_HOSPITAIS})",
        "Denominador": f"População total da regional ({ANO_REFERENCIA_HOSPITAIS})",
        "Agregação Extra": "Tipo de Habilitação",
    },
    {
        "Variável": "Coluna TOTAL de qualquer variável acima",
        "Numerador": "Soma das categorias daquele ano",
        "Denominador": "População total da regional",
        "Agregação Extra": "--",
    },
]

tabela_numerador_denominador = pd.DataFrame(linhas_tabela_taxa)

tabela_numerador_denominador_formatada = (
    tabela_numerador_denominador.style
    .set_caption("Numerador e denominador utilizados no cálculo da taxa por 100 mil habitantes, por variável")
    .set_properties(**{"text-align": "left", "white-space": "normal"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    .hide(axis="index")
)

tabela_numerador_denominador_formatada

Variável,Numerador,Denominador,Agregação Extra
Casos (Ano de Diagnóstico),"Casos, por ano (1985–2023)",População total da regional,--
Casos por Faixa Etária,"Casos, por ano e faixa etária (1985–2023)",População da mesma faixa etária e regional,--
Casos por Escolaridade,"Casos, por ano e escolaridade (1998–2023)",População total da regional,--
Casos por Raça e Cor,"Casos, por ano e raça/cor (1998–2023)",População total da regional,--
Mamografia por Faixa Etária,"Mamografias, por ano e faixa etária (2013–2025)",População da mesma faixa etária e regional,--
Mortalidade por Faixa Etária,"Óbitos, por ano e faixa etária (1979–2024)",População da mesma faixa etária e regional,--
Hospitais Habilitados,Hospitais habilitados (2023),População total da regional (2023),Tipo de Habilitação
Coluna TOTAL de qualquer variável acima,Soma das categorias daquele ano,População total da regional,--
